In [0]:
-- 3 HCOs not part of target
-- ('1093728743', '1184722779', '1861439952')

In [0]:
select distinct hco_npi
from cmpa_insights_internal_schema.reference_file_0109
where hco_npi_crosswalk in ('1093728743', '1184722779', '1861439952')

In [0]:
select distinct hco_npi_crosswalk, hco_name_crosswalk, hco_npi, hco_name from cmpa_insights_internal_schema.reference_file_0109
where hco_target != '-' or hco_npi in (select distinct hco_npi
from cmpa_insights_internal_schema.reference_file_0109
where hco_npi_crosswalk in ('1093728743', '1184722779', '1861439952'))

In [0]:
select distinct hco_npi_crosswalk, hco_name_crosswalk, hco_npi, hco_name from cmpa_insights_internal_schema.reference_file_0109
where hco_target != '-' or hco_npi in (select distinct hco_npi
from cmpa_insights_internal_schema.reference_file_0109
where hco_npi_crosswalk in ('1093728743', '1184722779', '1861439952'))

In [0]:
select *
from com_edp_prd.cmpa_insights_internal_schema.base_reference_file
where hco_npi in ('1083630073', '1679973364')

In [0]:
with target_hcos as (
  select distinct hco_npi from cmpa_insights_internal_schema.reference_file_0109
where hco_target != '-' or hco_npi in (select distinct hco_npi
from cmpa_insights_internal_schema.reference_file_0109
where hco_npi_crosswalk in ('1093728743', '1184722779', '1861439952'))
),
hco_zip_v1 AS (
    SELECT DISTINCT
        a.npi_num__v AS hco_npi,
        b.address_line_1__v as hco_address,
        b.postal_code_cda__v AS hco_postal_code,
        b.modified_date__v,
        ROW_NUMBER() OVER (
            PARTITION BY a.npi_num__v
            ORDER BY b.modified_date__v DESC
        ) AS rn
    FROM com_raw.vod_hco a
    JOIN com_raw.vod_address b
        ON b.entity_vid__v = a.vid__v
       AND b.entity_type__v = 'HCO'
       AND b.record_state__v = 'VALID'
       AND b.address_status__v IN ('A','DS')
       AND b.address_verification_status__v NOT IN ('NS','U')
    WHERE a.npi_num__v IN (
        SELECT DISTINCT hco_npi
        FROM target_hcos
    )
),
hco_zip_v2 AS (
    SELECT
        hco_npi,
        hco_address,
        hco_postal_code
    FROM hco_zip_v1
    WHERE rn = 1
),
pulling_hco_zip_using_vod_komodo AS (
    SELECT
        a.hco_npi,
        case when b.hco_postal_code is not null then b.hco_address else coalesce(c.provider_address, '-') end as hco_address,
        COALESCE(b.hco_postal_code, c.provider_zip, '-') AS hco_zip
    FROM target_hcos a
    LEFT JOIN hco_zip_v2 b
        ON a.hco_npi = b.hco_npi
    LEFT JOIN com_raw.kom_providers c
        ON a.hco_npi = c.npi
       AND c.provider_type = 'ORGANIZATION'
),
territory_region_reassignment AS (
    SELECT
        a.*,
        coalesce(b.territory_name, '-') AS territory,
        coalesce(b.region_name, '-') AS region,
        coalesce(b.state, '-') as state
    FROM pulling_hco_zip_using_vod_komodo a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping b
        ON TRY_CAST(NULLIF(a.hco_zip, '-') AS BIGINT) = b.zipcode
)
select distinct *
from territory_region_reassignment

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.reference_file_0109
where hco_npi_crosswalk in ('1093728743')

In [0]:
select *
from cmpa_insights_internal_schema.base_reference_file
where hco_target in ('1104819366')

In [0]:
select count(distinct hcp_npi) from 
cmpa_insights_internal_schema.reference_file_0109
where hco_npi_crosswalk != '-'

In [0]:
select * from 
cmpa_insights_internal_schema.reference_file_0109
where hco_npi_crosswalk in ('1932280666',
'1336495910',
'1477643690',
'1104819366')

In [0]:
select * from
com_edp_prd.cmpa_insights_internal_schema.secondary_to_primary_npi
where primary_npi in ('1932280666',
'1336495910',
'1477643690',
'1104819366')

In [0]:
select * from com_raw.vod_references